In [197]:
import random
import time
import math
import numpy as np

import deap
from deap import base, creator, tools

In [198]:
plantType = ["Null", "Corn", "Beans", "Squash"]

DEFAULT_NITROGEN = 50 # 50 ppm as baseline for the soil.
DEFAULT_LIGHT = 1 # 100% sunlight as baseline for the light level.
DEFAULT_YIELD = 0

CELLDISTANCE = 0

def create_plant(plantTypeVal):
    plantInd = [plantType[plantTypeVal], int(DEFAULT_NITROGEN), float(DEFAULT_LIGHT), float(DEFAULT_YIELD)]
    return plantInd

# Modifier Layout: [Plant-Corn Nitrogen, Plant-Corn Light, Plant-Beans Nitrogen, Plant-Beans Light, Plant-Squash Nitrogen, Plant-Squash Light]
CORN_MODIFIER = [-15, -.2, -10, .1, -5, -.25]
BEANS_MODIFIER = [20, 0, -5, -.1, 15, -.05]
SQUASH_MODIFIER = [5, 0, 5, -.05, -8, -.15]

def apply_plant_modifier(castingPlant, targetPlant):
    if (castingPlant[0] == "Null"):
        return targetPlant

    modifier = []
    if (castingPlant[0] == "Corn"):
        modifier = CORN_MODIFIER
    elif (castingPlant[0] == "Beans"):
        modifier = BEANS_MODIFIER
    elif (castingPlant[0] == "Squash"):
        modifier = SQUASH_MODIFIER

    if (targetPlant[0] == "Corn"):
        targetPlant[1] += modifier[0]
        targetPlant[2] += modifier[1]
    elif (targetPlant[0] == "Beans"):
        targetPlant[1] += modifier[2]
        targetPlant[2] += modifier[3]
    elif (targetPlant[0] == "Squash"):
        targetPlant[1] += modifier[4]
        targetPlant[2] += modifier[5]

    # Clamp nitrogen >= 0 and light to [0, 1]
    targetPlant[1] = max(0, targetPlant[1])
    targetPlant[2] = max(0.0, min(1.0, targetPlant[2]))

    return targetPlant

def calculate_all_modifiers(plantArray):
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            curPlant = plantArray[row][col]
            if (row != 0):
                apply_plant_modifier(plantArray[row-1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row-1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row-1][col+1], curPlant)
            if (row != plantArray.shape[0]-1):
                apply_plant_modifier(plantArray[row+1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row+1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row+1][col+1], curPlant)
            if (col != 0):
                apply_plant_modifier(plantArray[row][col-1], curPlant)
            if (col != plantArray.shape[1]-1):
                apply_plant_modifier(plantArray[row][col+1], curPlant)

def calculate_yield_val(value, optMin, optMax):
    mid = (optMin + optMax) / 2
    half_range = (optMax - optMin) / 2
    yieldVal = max(0, half_range - abs(value - mid))
    return yieldVal

CORN_NITROGEN_RANGE = [120, 200]
CORN_LIGHT_RANGE = [.8, 1]
BEANS_NITROGEN_RANGE = [30, 60]
BEANS_LIGHT_RANGE = [.5, .75]
SQUASH_NITROGEN_RANGE = [60, 120]
SQUASH_LIGHT_RANGE = [.65, .9]

def calculate_plant_yield(plantInd):
    if (plantInd[0] == "Null"):
        return plantInd

    nitrogenRange = []
    lightRange = []

    if (plantInd[0] == "Corn"):
        nitrogenRange = CORN_NITROGEN_RANGE
        lightRange = CORN_LIGHT_RANGE
    elif (plantInd[0] == "Beans"):
        nitrogenRange = BEANS_NITROGEN_RANGE
        lightRange = BEANS_LIGHT_RANGE
    elif (plantInd[0] == "Squash"):
        nitrogenRange = SQUASH_NITROGEN_RANGE
        lightRange = SQUASH_LIGHT_RANGE

    nitrogenValue = calculate_yield_val(plantInd[1], nitrogenRange[0], nitrogenRange[1])
    lightValue = calculate_yield_val(plantInd[2], lightRange[0], lightRange[1])

    plantInd[3] = nitrogenValue + lightValue
    return plantInd

def calculate_all_yields(plantArray):
    totalYield = 0
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            calculate_plant_yield(plantArray[row][col])
            totalYield += plantArray[row][col][3]
    return totalYield

In [199]:
# Test functions

def print_plant_array(plantArray):
    cols = len(plantArray[0])
    sep = "+" + ("-" * 16 + "+") * cols
    for row in plantArray:
        print(sep)
        cells = [f" {p[0]:<6} Y:{p[3]:>5.2f} " for p in row]
        print("|" + "|".join(cells) + "|")
    print(sep)

def print_full_plant_array(plantArray):
    cols = len(plantArray[0])
    sep = "+" + ("-" * 32 + "+") * cols
    for row in plantArray:
        print(sep)
        cells = [f" {p[0]:<6} N:{p[1]:>4}  L:{p[2]:.2f}  Y:{p[3]:>5.2f} " for p in row]
        print("|" + "|".join(cells) + "|")
    print(sep)



def test_yields(xSize, ySize):
    fieldArray = np.array([[create_plant(random.randint(0, 3)) for _ in range(ySize)] for _ in range(xSize)], dtype=object)
    calculate_all_modifiers(fieldArray)
    totalYield = calculate_all_yields(fieldArray)
    print_full_plant_array(fieldArray)
    print(f"Total Yield: {totalYield}")

test_yields(5, 5)
    

+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  37  L:0.60  Y: 0.00 | Corn   N:  85  L:1.00  Y: 0.00 | Beans  N:  55  L:0.95  Y: 5.00 | Null   N:  50  L:1.00  Y: 0.00 | Beans  N:  55  L:0.95  Y: 5.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Null   N:  50  L:1.00  Y: 0.00 | Squash N:  23  L:0.00  Y: 0.00 | Squash N:  28  L:0.10  Y: 0.00 | Squash N:  59  L:0.35  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Corn   N:  65  L:1.00  Y: 0.00 | Squash N:   5  L:0.00  Y: 0.00 | Squash N:   2  L:0.10  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Corn   N:  60  L:1.00  Y: 0.00 

In [200]:
GRID_X = 5
GRID_Y = 5
N_GENES = GRID_X * GRID_Y

POP_SIZE = 100

def evaluate(individual):
    plantArray = np.array([[create_plant(individual[row * GRID_Y + col]) for col in range(GRID_Y)] for row in range(GRID_X)], dtype=object)
    calculate_all_modifiers(plantArray)
    totalYield = calculate_all_yields(plantArray)
    return (totalYield,)

In [201]:
def run_random():
    history = []
    bestHistory = []
    bestYield = -1
    for i in range(POP_SIZE):
        fieldArray = np.array([[create_plant(random.randint(0, 3)) for _ in range(GRID_Y)] for _ in range(GRID_X)], dtype=object)
        calculate_all_modifiers(fieldArray)
        totalYield = calculate_all_yields(fieldArray)
        history.append(totalYield)
        if (totalYield > bestYield):
            bestYield = totalYield
            bestArray = fieldArray
        bestHistory.append(bestYield)
    return bestArray, history, bestHistory

In [202]:
def run_greedy():
    history = []
    bestHistory = []
    bestArray = []
    bestYield = -1
    for i in range(POP_SIZE):
        fieldArray = [0] * N_GENES
        for j in range(N_GENES):
            bestGreedyPlant = -1
            bestPlantYield = -1
            for plantType in range(4):
                fieldArray[j] = plantType
                curPlantYield = evaluate(fieldArray)
                if (curPlantYield[0] > bestPlantYield):
                    bestPlantYield = curPlantYield[0]
                    bestGreedyPlant = plantType
            fieldArray[j] = bestGreedyPlant

        individual = np.array([[create_plant(fieldArray[row * GRID_Y + col]) for col in range (GRID_Y)] for row in range (GRID_X)], dtype=object)
        calculate_all_modifiers(individual)
        totalYield = calculate_all_yields(individual)
        history.append(totalYield)
        if (totalYield > bestYield):
            bestYield = totalYield
            bestArray = individual
        bestHistory.append(bestYield)
    return bestArray, history, bestHistory

In [203]:
MAX_PASSES = 25

def run_hillclimb():
    history = []
    bestHistory = []
    max_passes = MAX_PASSES
    bestYield = -1
    bestIndividual = []
    for i in range(POP_SIZE):
        moreImprovements = True
        fieldArray = np.random.randint(0, 4, size=(N_GENES))
        passes = 0
        while moreImprovements:
            moreImprovements = False
            for j in range(N_GENES):
                bestGreedyPlant = -1
                bestPlantYield = -1
                for plantType in range(4):
                    fieldArray[j] = plantType
                    curPlantYield = evaluate(fieldArray)
                    if (curPlantYield[0] > bestPlantYield):
                        bestPlantYield = curPlantYield[0]
                        bestGreedyPlant = plantType
                        moreImprovements = True
                fieldArray[j] = bestGreedyPlant
            passes = passes + 1
            if (passes >= max_passes):
                moreImprovements = False
        history.append(bestPlantYield)
        if (bestYield < bestPlantYield):
            bestYield = bestPlantYield
            bestIndividual = fieldArray
        bestHistory.append(bestYield)
    individual = np.array([[create_plant(bestIndividual[row * GRID_Y + col]) for col in range (GRID_Y)] for row in range (GRID_X)], dtype=object)
    calculate_all_modifiers(individual)
    return individual, history, bestHistory

In [204]:
def run_ga():
    if not hasattr(creator, "FitnessMax"):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()

    toolbox.register("attr_plant", random.randint, 0, 3)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_plant, n=N_GENES)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutUniformInt, low=0, up=3, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)
    
    POP_SIZE = 100
    N_GEN = 100
    CROSSOVER = 0.7
    MUTATION = 0.2

    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    hof = tools.HallOfFame(1)
    hof.update(pop)

    bestHistory, meanHistory = [], []

    for gen in range(1, N_GEN + 1):
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        # Crossover
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CROSSOVER:
                toolbox.mate(c1, c2)
                del c1.fitness.values
                del c2.fitness.values

        # Mutation
        for mutant in offspring:
            if random.random() < MUTATION:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid:
            ind.fitness.values = toolbox.evaluate(ind)

        pop[:] = offspring

        hof.update(pop)

        fitness = [ind.fitness.values[0] for ind in pop]

        bestHistory.append(max(fitness))
        meanHistory.append(np.mean(fitness))
        # print(f"{gen:>4} {np.mean(fitness):>10.4f} {np.max(fitness):>10.4f}")

    best = hof[0]
    bestArray = np.array([[create_plant(best[row * GRID_Y + col]) for col in range(GRID_Y)] for row in range(GRID_X)], dtype=object)
    calculate_all_modifiers(bestArray)
    bestYield = calculate_all_yields(bestArray)
    print_full_plant_array(bestArray)
    print(f"Best Yield: {bestYield:.4f}")

    return bestArray, meanHistory, bestHistory

# continue from here

In [205]:
import matplotlib.pyplot as plt
import os

def plot_algorithm(history, title, xLabel, meanHistory = None, meanPlotLabel = "Population Mean", outputDir="plots"):
    os.makedirs(outputDir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(9,5))
    ax.plot(history, label="Best", linewidth=2)
    if meanHistory is not None:
        ax.plot(meanHistory, label=meanPlotLabel, linewidth=2, linestyle="--", alpha=0.8)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(xLabel)
    ax.set_ylabel("Yield")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    filename = title.lower().replace(" ", "_") + ".png"
    fig.savefig(os.path.join(outputDir, filename), dpi=150)
    plt.close(fig)

In [206]:
import json

def build_result(algorithm, bestYield, bestArray):
    genes = [bestArray[row][col][0] for row in range(GRID_X) for col in range (GRID_Y)]
    plantToInt = {"Null": 0, "Corn": 1, "Beans": 2, "Squash": 3}
    return {
        "algorithm": algorithm,
        "best_yield": bestYield,
        "grid_x": GRID_X,
        "grid_y": GRID_Y,
        "genes": genes
    }

def log_results(entries, filepath):
    try:
        with open(filepath, "r") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        data = {"results": []}
    data["results"].extend(entries)
    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)

In [207]:
def main():
    bestRandomInd, randomHistory, randomBestHist = run_random()
    bestGreedyInd, greedyHistory, greedyBestHist = run_greedy()
    bestHillInd, hillclimbHistory, hillclimbBestHist = run_hillclimb()
    bestGAInd, gaMean, gaBest = run_ga()

    plot_algorithm(randomBestHist, "Random Search", xLabel="Sample #", meanPlotLabel="Result", meanHistory=randomHistory)
    plot_algorithm(greedyBestHist, "Greedy Search", xLabel="Restart #", meanPlotLabel="Result", meanHistory=greedyHistory)
    plot_algorithm(hillclimbBestHist, "Hill Climb", xLabel="Restart #", meanPlotLabel="Result", meanHistory=hillclimbHistory)
    plot_algorithm(gaBest, "Genetic Algorithm", xLabel="Generation", meanHistory=gaMean)

    entries = [
        build_result("Random Search", randomHistory[-1], bestRandomInd),
        build_result("Greedy Search", greedyHistory[-1], bestGreedyInd),
        build_result("Hill Climb", hillclimbHistory[-1], bestHillInd),
        build_result("Genetic Algorithm", gaBest[-1], bestGAInd)
    ]
    log_results(entries, filepath="results.json")

main()

+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  95  L:0.85  Y:25.05 | Beans  N:  40  L:0.80  Y:10.00 | Squash N: 105  L:0.55  Y:15.00 | Beans  N:  40  L:0.80  Y:10.00 | Squash N:  95  L:0.85  Y:25.05 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Beans  N:  45  L:0.60  Y:15.10 | Beans  N:  45  L:0.60  Y:15.10 | Corn   N: 165  L:1.00  Y:35.00 | Beans  N:  45  L:0.60  Y:15.10 | Beans  N:  45  L:0.60  Y:15.10 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  82  L:0.45  Y:22.00 | Beans  N:  30  L:0.75  Y: 0.00 | Squash N:  84  L:0.00  Y:24.00 | Squash N: 104  L:0.20  Y:16.00 | Beans  N:  45  L:0.60  Y:15.10 